# LLM From Scratch + Post-Training + Agentic RAG — Colab
Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.

**Run all is always safe.** Every step checks whether its output already exists (in this session, on Google Drive or on the Hugging Face Hub) and prints `[skip]` instead of redoing it. After a disconnect or a quota cut-off, reconnect and press **Run all** again: it continues where it stopped. Evaluation results are saved to Drive after every single experiment.

One-time setup: add your Hugging Face write token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
import os
REPO = '/content/llm-from-scratch-agentic-rag'
# New session -> clone. Same session -> update to the latest code (local edits are discarded;
# all results live on Drive and are restored below).
if os.path.exists(REPO):
    %cd {REPO}
    !git fetch -q && git reset -q --hard origin/main
else:
    !git clone -q https://github.com/ahmedashrafk13/llm-from-scratch-agentic-rag.git {REPO}
    %cd {REPO}
!pip install -q transformers==4.46.3 datasets==3.1.0 peft==0.13.2 bitsandbytes accelerate sentence-transformers==3.3.1 faiss-cpu rank_bm25 regex gradio

from google.colab import userdata, drive
from huggingface_hub import login, whoami
try:
    login(userdata.get('HF_TOKEN'))
except Exception:
    print('HF_TOKEN secret not found - paste your token below')
    login()
HF_USER = whoami()['name']
HUB = f'{HF_USER}/llm-scratch-checkpoints'   # private: resumable pre-training state + eval results
MODELS = f'{HF_USER}/llm-scratch-37m'        # public: trained models + tokenizer

drive.mount('/content/drive')
D = '/content/drive/MyDrive/llm-scratch'
os.makedirs(f'{D}/results', exist_ok=True)
os.environ['RESULTS_MIRROR'] = f'{D}/results'   # eval scripts copy results here after every experiment
os.makedirs('results', exist_ok=True)
# restore results from Drive; files already in the repo (final, committed results) take precedence
!cp -rn {D}/results/. results/ 2>/dev/null; true

def step(name, done, cmd):
    """Run a shell command unless its output already exists. Stops Run all on failure."""
    if done:
        print(f'[skip] {name} (already done)')
        return
    print(f'[run ] {name}', flush=True)
    get_ipython().system(cmd)
    if get_ipython().user_ns.get('_exit_code', 0):
        raise RuntimeError(f'{name} failed. Fix the error, then Run all again - finished steps are skipped.')

print('ready:', HF_USER)

In [ ]:
step('unit tests', False, 'python -m pytest tests -q')

## 1. Data + tokenizer
Each step is saved to Drive when it finishes and skipped next time. The tokenizer is in the repo (`tokenizer/tokenizer.json`), so it is never retrained.

In [ ]:
P = 'data/processed'
os.makedirs(f'{D}/processed', exist_ok=True); os.makedirs(P, exist_ok=True)
!cp -rn {D}/processed/. {P}/ 2>/dev/null; true

def data_step(name, done_file, cmd):
    step(name, os.path.exists(done_file), cmd)
    get_ipython().system(f'cp -rn {P}/. {D}/processed/')

data_step('download + clean',   f'{P}/fineweb.jsonl',                     'python data/prepare_pretrain.py')
data_step('train tokenizer',    'tokenizer/tokenizer.json',               'python tokenizer/train_tokenizer.py')
data_step('tokenize corpus',    f'{P}/val_fineweb.bin',                   'python data/tokenize_corpus.py')
data_step('build SFT/RAG data', f'{P}/domain_eval.jsonl',                 'python data/build_sft.py')
data_step('hallucination set',  'evaluation/hallucination_testset.jsonl', 'python evaluation/build_hallucination_set.py')

## 2. Restore trained models
Trained checkpoints are pulled from the Hub (or Drive), so nothing below retrains them.

In [ ]:
import shutil
from huggingface_hub import list_repo_files, snapshot_download
try:
    hub_files = set(list_repo_files(MODELS))
except Exception:
    hub_files = set()
for d in ('base', 'sft', 'dpo', 'qlora'):
    if any(os.path.exists(f'checkpoints/{d}/{f}') for f in ('model.pt', 'adapter_model.safetensors')):
        print(f'[skip] {d} (present)')
    elif any(f.startswith(f'{d}/') for f in hub_files):
        p = snapshot_download(MODELS, allow_patterns=[f'{d}/*'])
        shutil.copytree(f'{p}/{d}', f'checkpoints/{d}', dirs_exist_ok=True)
        print(f'[hub ] restored {d}')
    elif os.path.exists(f'{D}/checkpoints/{d}'):
        shutil.copytree(f'{D}/checkpoints/{d}', f'checkpoints/{d}', dirs_exist_ok=True)
        print(f'[drive] restored {d}')
    else:
        print(f'[todo] {d} - will be trained below')

## 3. Pre-training (~1 h on T4)
Skipped when the base model exists. Otherwise `--resume` continues from the last Hub checkpoint.

In [ ]:
step('pre-training', os.path.exists('checkpoints/base/model.pt'),
     f'python training/pretrain.py --batch 32 --accum 1 --resume --hub-repo {HUB}')

## 4. SFT → DPO → reward model (~20 min)

In [ ]:
step('SFT',          os.path.exists('checkpoints/sft/model.pt'),  'python training/sft.py')
step('DPO',          os.path.exists('checkpoints/dpo/model.pt'),  'python training/dpo.py')
step('reward model', os.path.exists('results/reward_model.json'), 'python training/reward_model.py')
!mkdir -p {D}/checkpoints && cp -rn checkpoints/. {D}/checkpoints/

## 5. QLoRA on Qwen2.5-1.5B (~35 min)

In [ ]:
step('QLoRA', os.path.exists('checkpoints/qlora/adapter_model.safetensors'), 'python training/qlora.py')
!mkdir -p {D}/checkpoints && cp -rn checkpoints/. {D}/checkpoints/

## 6. Evaluation (~2 h on T4 if nothing is done yet)
Every script is resumable: finished experiments are skipped and each new result is saved to Drive immediately. Before each script the cell pulls results already committed to the repo, so work finished elsewhere is not repeated.

In [ ]:
from huggingface_hub import HfApi

def sync_committed_results():
    get_ipython().system('git fetch -q && git checkout -q origin/main -- results 2>/dev/null; true')

def push_results():  # private Hub copy, so results can be pulled without opening Drive
    HfApi().upload_folder(folder_path='results', repo_id=HUB, path_in_repo='results',
                          allow_patterns=['*.json', '*.jsonl', '*.csv', '*.md', '*.png'], commit_message='eval results')

sync_committed_results()
step('QA experiments', False, 'python evaluation/qa_eval.py --n 200')
push_results()
sync_committed_results()
step('post-training eval',   os.path.exists('results/post_training_eval.json'), 'python evaluation/post_training_eval.py')
step('pre-training summary', os.path.exists('results/pretrain_summary.json'),   'python evaluation/pretrain_summary.py')
step('benchmarks', False, 'python evaluation/benchmarks.py')
push_results()
sync_committed_results()
step('retrieval ablations', False, 'python evaluation/retrieval_eval.py')
step('report', False, 'python evaluation/make_report.py')
!cp -r results/. {D}/results/
push_results()
print('evaluation complete')

## 7. Publish models to the Hub
Uploads only what is missing (including the tokenizer, which the models need).

In [ ]:
api = HfApi()
api.create_repo(MODELS, exist_ok=True)
have = set(api.list_repo_files(MODELS))
todo = [d for d in ('base', 'sft', 'dpo', 'qlora') if not any(f.startswith(f'{d}/') for f in have)]
if todo:
    api.upload_folder(folder_path='checkpoints', repo_id=MODELS, allow_patterns=[f'{d}/*' for d in todo])
if 'tokenizer.json' not in have:
    api.upload_file(path_or_fileobj='tokenizer/tokenizer.json', path_in_repo='tokenizer.json', repo_id=MODELS)
    todo.append('tokenizer.json')
print('uploaded:', todo or 'nothing (all present)')